# Clase 5 — Visión: clasificación, detección y segmentación

## Pregunta central

> **¿Qué puede responder un modelo cuando recibe una imagen?**

## Idea principal

Una imagen es un tensor; según la tarea, el modelo devuelve una clase, cajas o una máscara por píxel.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Distinguir clasificación, detección y segmentación por sus salidas.
- Explicar el rol de una CNN, el preprocesamiento y el augmentation.
- Usar ResNet18 y YOLO nano preentrenados sin entrenarlos.
- Interpretar boxes, masks, IoU y las métricas principales.
- Ubicar ResNet, EfficientNet, YOLO, Faster R-CNN, U-Net, Mask R-CNN y SAM.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Tres preguntas visuales diferentes |
| 2 | Imágenes, CNN y preparación de datos |
| 3 | Modelos preentrenados y transfer learning |
| 4 | Práctica: ResNet18, augmentation y YOLO nano |
| 5 | Cajas, máscaras y métricas |
| 6 | Actividad de interpretación |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** Clases 1 y 2 del Track Imagen: clasificación, detección y segmentación.

---
## 1. Tres preguntas visuales diferentes

En visión por computadora, un **modelo** es una función que recibe
números que representan una imagen y produce una salida. Usar un
modelo ya preparado para obtener esa salida se llama **inferencia**.

Antes de elegir una arquitectura debemos escribir qué respuesta
necesita el sistema:

- una **clase** es el nombre de una categoría, por ejemplo `perro`;
- un **score** es un valor numérico asociado con una predicción;
- una **caja** es un rectángulo que aproxima dónde está un objeto;
- una **máscara** asigna una marca a cada píxel de una región.

Una misma imagen admite preguntas diferentes. Por eso la **salida
esperada** define qué anotaciones, métricas y modelo necesitamos.

| Tarea | Pregunta | Salida típica | Ejemplo |
|---|---|---|---|
| Clasificación | ¿Qué predomina en la imagen? | Clase + score | `cultivo sano: 0.87` |
| Detección | ¿Qué objetos hay y dónde? | Clase + caja + score | `vehículo, [x1,y1,x2,y2]` |
| Segmentación semántica | ¿Qué clase tiene cada píxel? | Una máscara por clase | cultivo/suelo/agua |
| Segmentación de instancias | ¿Qué píxeles pertenecen a cada objeto? | Una máscara por objeto | árbol 1, árbol 2 |

```text
imagen ── clasificación ──> "perro"
       ├─ detección ──────> "perro" + rectángulo
       └─ segmentación ───> píxeles que forman el perro
```

En clasificación, toda la imagen recibe una respuesta. En
detección puede haber cero, uno o muchos objetos. En segmentación
la salida conserva la estructura espacial de la entrada.

Una **bounding box** o caja delimitadora aproxima una ubicación con
un rectángulo. Una **mask** o máscara puede seguir el contorno del
objeto. La máscara contiene más detalle, pero etiquetarla y
procesarla suele costar más.

> No existe una tarea “mejor” en general. Si solo necesitamos saber
> si hay humo, clasificar puede alcanzar. Si además necesitamos
> ubicarlo, hace falta detección o segmentación.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Píxel | Valor numérico de una posición de la imagen |
| Canal | Componente de información, por ejemplo R, G y B |
| Tensor | Arreglo numérico con una forma, un tipo y un rango |
| Label | Respuesta correcta asignada a una muestra |
| Anotación | Label, caja o máscara creada para entrenar o evaluar |
| Feature | Patrón útil: borde, textura, forma o combinación |
| CNN | Red que aplica filtros compartidos sobre regiones locales |
| Backbone | Parte que extrae features |
| Head | Parte que produce la salida de una tarea |
| Ground truth | Etiqueta, caja o máscara considerada correcta |
| Score | Confianza numérica del modelo; no es garantía de verdad |
| Threshold | Umbral usado para aceptar o descartar una salida |

---
## 2. De píxeles a features: qué hace una CNN

### Una imagen también es un conjunto de números

Un **píxel** es una posición de la grilla. En una imagen RGB, cada
píxel contiene tres intensidades: rojo, verde y azul. Una imagen de
`640 × 427` no contiene 640 números, sino:

```text
427 filas × 640 columnas × 3 canales = 819 840 valores
```

En archivos comunes esos valores suelen ser enteros de 0 a 255
(`uint8`). Para una red suelen convertirse a números decimales
(`float32`) y cambiar de escala.

```text
canal R ─┐
canal G ─┼─> tensor de la imagen
canal B ─┘
```

Un **tensor** es el contenedor numérico usado por la red. Para
interpretarlo no alcanza con mirar sus valores: necesitamos saber
su `shape` (forma), `dtype` (tipo numérico) y rango.

Según la librería, la forma puede verse como:

```text
(alto, ancho, canales)       # NumPy / PIL
(lote, canales, alto, ancho) # PyTorch
```

Un **lote** o batch agrupa varias imágenes para procesarlas juntas.
Por ejemplo, `(8, 3, 224, 224)` significa 8 imágenes RGB de
224 × 224 píxeles.

Las coordenadas suelen comenzar arriba a la izquierda:

```text
(0,0) ─────────────> x / columna
  |
  |
  v
y / fila
```

### Convolución, paso a paso

Una **CNN** (Convolutional Neural Network) usa una operación llamada
convolución. Un **kernel** o filtro es una matriz pequeña, por
ejemplo de `3 × 3`. La red desplaza ese filtro por la imagen:

```text
región 3 × 3 de la imagen
           × kernel aprendido
           ↓
     suma ponderada
           ↓
  un valor del feature map
```

El resultado completo es un **feature map**: un mapa que indica
dónde respondió ese filtro. Los mismos nueve pesos se reutilizan en
todas las posiciones. Esta reutilización permite detectar un patrón
aunque aparezca en otra zona y necesita menos parámetros que
conectar cada píxel con todos los demás.

Después de una convolución suelen aparecer:

- una **activación**, como ReLU, que permite relaciones no lineales;
- **pooling** o reducción espacial, que resume regiones;
- nuevas convoluciones que combinan patrones anteriores.

La zona de la imagen que influye en una activación se llama
**campo receptivo**. Al apilar capas, ese campo crece: una capa
temprana puede responder a un borde y una capa posterior a una
combinación de partes.

Durante el **entrenamiento**, la red ajusta los valores de sus
filtros usando ejemplos y respuestas correctas. Durante la
**inferencia**, conserva esos valores y solo calcula una salida:

```text
píxeles → bordes → texturas → partes → representación visual
                                        ↓
                                head de la tarea
```

No significa que la red “vea como una persona”: transforma números
hasta obtener una salida útil para la tarea. La secuencia anterior
es una intuición, no una garantía de que cada neurona tenga una
interpretación humana única.

### Preprocesamiento y augmentation no son lo mismo

El **preprocesamiento** es la transformación estable que hace
compatible una entrada con el modelo. Debe respetar el contrato con
el que se entrenaron los pesos:

| Paso | Qué cambia | Por qué importa |
|---|---|---|
| Convertir a RGB | Cantidad y orden de canales | El modelo espera tres canales concretos |
| Redimensionar (resize) | Resolución | Permite trabajar con un tamaño conocido |
| Recortar (crop) | Región visible | Ajusta proporción y tamaño final |
| Convertir a tensor | Forma y tipo | La red opera con tensores, no con archivos JPEG |
| Normalizar | Escala de cada canal | Debe coincidir con la usada al preentrenar |

Normalizar no significa necesariamente llevar todo a 0–1. Algunos
modelos restan una media y dividen por un desvío específico. Si
cambiamos este contrato, los pesos reciben una distribución
diferente de la que aprendieron.

**Data augmentation** crea variaciones razonables durante
entrenamiento. Un mismo ejemplo puede presentarse con giros
pequeños, flips, crops o cambios de color. Busca que el modelo
dependa menos de detalles accidentales y aprenda invariancias
válidas para el dominio.

> Una transformación solo es válida si conserva el significado.
> Un flip horizontal puede ser razonable para animales y peligroso
> para texto, señales o estructuras con orientación.

Si transformamos una imagen anotada, sus anotaciones espaciales
deben cambiar junto con ella:

```text
imagen + caja + máscara
        ↓ misma transformación geométrica
imagen' + caja' + máscara'
```

Girar solo la imagen dejaría la caja o la máscara en una posición
incorrecta.

Validación, test e inferencia deben usar transformaciones
deterministas y coherentes. No se “mejora” el test cambiándolo al
azar: necesitamos una referencia estable para comparar modelos.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.datasets import load_sample_image, load_sample_images
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import functional as TF

FAST_MODE = True
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

def encontrar_imagen(nombre):
    candidatos = [
        Path.cwd() / "assets" / "images" / nombre,
        (
            Path.cwd()
            / "IA para programadores"
            / "assets"
            / "images"
            / nombre
        ),
    ]
    for padre in Path.cwd().parents:
        candidatos.append(
            padre
            / "IA para programadores"
            / "assets"
            / "images"
            / nombre
        )
    return next(
        (ruta.resolve() for ruta in candidatos if ruta.exists()),
        None,
    )

ruta_china = encontrar_imagen("china.jpg")
ruta_flor = encontrar_imagen("flower.jpg")
ruta_deteccion = encontrar_imagen("escena_bus_peatones.jpg")

imagenes = {
    "paisaje": (
        Image.open(ruta_china).convert("RGB")
        if ruta_china
        else Image.fromarray(
            load_sample_image("china.jpg")
        ).convert("RGB")
    ),
    "flor": (
        Image.open(ruta_flor).convert("RGB")
        if ruta_flor
        else Image.fromarray(
            load_sample_image("flower.jpg")
        ).convert("RGB")
    ),
}

if ruta_deteccion:
    imagenes["deteccion_calle"] = Image.open(
        ruta_deteccion
    ).convert("RGB")

print("FAST_MODE:", FAST_MODE)
print("PyTorch:", torch.__version__)
print("Imágenes disponibles:", list(imagenes))

### Origen y licencia de los recursos

- `assets/images/china.jpg` y `flower.jpg` son copias locales de
  `sklearn.datasets.load_sample_images`. Sus autores los publicaron
  bajo **CC BY 2.0**; la atribución y los enlaces originales están
  en `load_sample_images().DESCR`.
- `assets/images/escena_bus_peatones.jpg` fue generada
  específicamente para este curso. No representa personas, marcas
  ni vehículos reales y permite observar detecciones sin depender
  de una fotografía externa.
- ResNet18 se obtiene mediante los pesos oficiales de Torchvision.
- YOLO11n se obtiene mediante Ultralytics. Código y pesos están
  sujetos a la licencia de
  [Ultralytics](https://www.ultralytics.com/license); antes de una
  distribución comercial corresponde revisar sus términos.

Para ver la atribución completa de las dos imágenes Creative
Commons:

In [ ]:
print(load_sample_images().DESCR)

fig, axes = plt.subplots(
    1, len(imagenes), figsize=(5 * len(imagenes), 4)
)
if len(imagenes) == 1:
    axes = [axes]
for ax, (nombre, imagen) in zip(axes, imagenes.items()):
    ax.imshow(imagen)
    ax.set_title(
        f"{nombre}\n{imagen.width} × {imagen.height} × 3"
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## Práctica A — visualizar augmentation

Mostramos transformaciones **fijas** para poder comparar. Durante
entrenamiento suelen sortearse variantes nuevas en cada **epoch**
o recorrido completo por los datos.

La pregunta importante no es “¿la imagen se ve distinta?”, sino
“¿la respuesta correcta debería seguir siendo la misma?”. Si la
respuesta cambia, esa transformación no sirve como augmentation
para esa tarea.

Ninguna de estas imágenes se guarda: son tensores o imágenes
generadas en memoria.

In [ ]:
imagen_base = imagenes["flor"]
variantes = {
    "original": imagen_base,
    "flip horizontal": TF.hflip(imagen_base),
    "rotación +20°": TF.rotate(imagen_base, angle=20),
    "menos brillo": TF.adjust_brightness(imagen_base, 0.55),
    "más contraste": TF.adjust_contrast(imagen_base, 1.7),
}

fig, axes = plt.subplots(1, len(variantes), figsize=(17, 4))
for ax, (nombre, variante) in zip(axes, variantes.items()):
    ax.imshow(variante)
    ax.set_title(nombre)
    ax.axis("off")
plt.suptitle(
    "Variaciones posibles de una misma muestra",
    y=1.03,
    fontsize=14,
)
plt.tight_layout()
plt.show()

### Preguntas de observación

1. ¿Las cinco variantes deberían conservar el mismo label?
2. ¿Qué transformación sería peligrosa para una radiografía?
3. ¿Por qué no aplicaríamos estas variaciones al azar en producción?
4. ¿Qué parte del proceso debe ser idéntica entre validación e
   inferencia?

---
## 3. Modelos preentrenados y transfer learning

Una **arquitectura** describe cómo están conectadas las capas. Los
**pesos** son los valores numéricos aprendidos por esas capas. Dos
modelos pueden compartir arquitectura y tener pesos distintos.

```text
arquitectura + pesos aprendidos = modelo listo para inferencia
```

Un modelo **preentrenado** ya ajustó sus pesos usando un
**dataset**, es decir, una colección organizada de ejemplos.
No conoce automáticamente nuestro problema, pero suele
haber aprendido features visuales reutilizables:

```text
imagen → backbone preentrenado → features → head de la tarea
```

El **backbone** transforma la imagen en una representación. El
**head** convierte esa representación en la salida específica:
clases, cajas o máscaras.

Hay tres estrategias generales:

| Estrategia | Qué se aprende | Cuándo considerarla |
|---|---|---|
| Entrenar desde cero | Todos los pesos parten al azar | Muchos datos y dominio muy particular |
| Feature extractor | Backbone congelado; aprende el head | Pocos datos o primer punto de comparación |
| Fine-tuning | Se ajusta también parte del backbone | Más datos y necesidad de adaptación |

**Congelar** una capa significa mantener sus pesos sin cambios.
**Fine-tuning** significa continuar ajustando pesos preentrenados
con datos propios y pasos pequeños.

Un flujo habitual de transfer learning es:

1. cargamos pesos preentrenados;
2. reemplazamos la salida por las clases propias;
3. congelamos inicialmente el backbone;
4. entrenamos la nueva salida;
5. si hay datos y cómputo, descongelamos algunas capas y hacemos
   fine-tuning con un learning rate bajo, es decir, actualizaciones
   pequeñas de los pesos.

Preentrenado no significa universal. Si el dataset original contiene
fotografías cotidianas y nuestro dominio contiene imágenes
satelitales multibanda, existe una **brecha de dominio**. Debemos
medir el resultado con datos propios.

Hoy solo hacemos inferencia. El entrenamiento, la adaptación y la
selección de qué capas ajustar pertenecen al Track Imagen.

### Mapa de familias: no compiten todas por la misma tarea

Los nombres siguientes identifican **familias de arquitecturas**.
Una familia puede tener versiones de distinto tamaño. La elección
empieza por la salida requerida, no por cuál nombre es más popular.

| Familia | Tarea habitual | Salida | Idea introductoria |
|---|---|---|---|
| ResNet | Clasificación / backbone | Vector de clases o features | Conexiones residuales dejan rutas cortas para la información |
| EfficientNet | Clasificación / backbone | Vector de clases o features | Escala en conjunto profundidad, ancho y resolución |
| YOLO | Detección de una etapa | Cajas, clases y scores | Produce candidatos en una sola pasada |
| Faster R-CNN | Detección de dos etapas | Cajas, clases y scores | Propone regiones y después las clasifica |
| U-Net | Segmentación semántica | Clase por píxel | Reduce y luego recupera resolución con conexiones entre escalas |
| Mask R-CNN | Segmentación de instancias | Caja y máscara por objeto | Extiende detección con un head de máscaras |
| SAM | Segmentación guiada | Máscaras candidatas | Recibe una guía como punto o caja |

En una arquitectura de tipo **encoder-decoder**, el encoder resume
información y el decoder recupera una salida espacial. Una
**conexión residual** suma o deriva información por una ruta
alternativa para facilitar redes profundas.

- **YOLO** suele priorizar velocidad.
- **Faster R-CNN** representa el enfoque de propuestas de regiones.
- **U-Net** responde por clase de píxel.
- **Mask R-CNN** separa instancias.
- **SAM** usa un prompt visual, como un punto o una caja, para
  proponer máscaras. Aquí “prompt” no implica necesariamente texto.

Tamaño, latencia, memoria y calidad dependen de la versión, los
datos y el hardware. “Nano” describe una variante pequeña; no
garantiza por sí mismo ni velocidad suficiente ni calidad adecuada.

---
## 4. Práctica B — clasificación con ResNet18

**ResNet18** es una CNN de la familia ResNet. El número 18 refiere
a su profundidad aproximada en capas con parámetros. Los pesos que
usaremos fueron entrenados para distinguir 1000 categorías de
ImageNet.

El flujo de inferencia será:

```text
imagen PIL
    ↓ resize + crop + tensor + normalización
tensor (3, 224, 224)
    ↓ ResNet18
1000 logits
    ↓ softmax
1000 scores que suman 1
    ↓ top-k
las k categorías con mayor score
```

Un **logit** es un puntaje de salida sin normalizar. **Softmax**
transforma los 1000 logits en valores no negativos que suman uno.
**Top-k** ordena esos valores y conserva las `k` alternativas más
altas. Ninguno de estos pasos verifica que la imagen pertenezca a
una de las 1000 categorías.

Torchvision publica junto a los pesos el preprocesamiento correcto.
Usarlo evita adivinar tamaño, recorte y normalización.

La primera ejecución puede descargar unos 45 MB. Si no hay red ni
cache, el notebook conserva la arquitectura y avisa que sus
predicciones no tienen valor semántico.

In [ ]:
pesos_resnet = ResNet18_Weights.DEFAULT
preprocesar_resnet = pesos_resnet.transforms()
categorias_imagenet = pesos_resnet.meta["categories"]

try:
    clasificador = resnet18(weights=pesos_resnet)
    resnet_preentrenada = True
except Exception as error:
    print("AVISO: no se pudieron cargar los pesos preentrenados.")
    print("Se usará la arquitectura sin entrenar:", error)
    clasificador = resnet18(weights=None)
    resnet_preentrenada = False

clasificador.eval()

def predecir_top_k(imagen, k=5):
    lote = preprocesar_resnet(imagen).unsqueeze(0)
    with torch.inference_mode():
        scores = clasificador(lote)
        probabilidades = scores.softmax(dim=1)[0]
    valores, indices = probabilidades.topk(k)
    return pd.DataFrame({
        "clase": [
            categorias_imagenet[indice]
            for indice in indices.tolist()
        ],
        "score": valores.cpu().tolist(),
    })

nombre_clasificacion = "flor"
imagen_clasificacion = imagenes[nombre_clasificacion]
ranking_resnet = predecir_top_k(imagen_clasificacion)

print("Imagen:", nombre_clasificacion)
print("Pesos preentrenados:", resnet_preentrenada)
display(ranking_resnet.style.format({"score": "{:.3f}"}))

In [ ]:
tensor_preprocesado = preprocesar_resnet(imagen_clasificacion)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(imagen_clasificacion)
axes[0].set_title("Entrada original")
axes[0].axis("off")

axes[1].barh(
    ranking_resnet["clase"][::-1],
    ranking_resnet["score"][::-1],
    color="#4472c4",
)
axes[1].set_xlim(0, max(1.0, ranking_resnet["score"].max() * 1.1))
axes[1].set_xlabel("score después de softmax")
axes[1].set_title("Top-5 de ResNet18")
plt.tight_layout()
plt.show()

print("Shape esperada por PyTorch:", tuple(tensor_preprocesado.shape))
print(
    "Rango después de normalizar:",
    round(float(tensor_preprocesado.min()), 2),
    "a",
    round(float(tensor_preprocesado.max()), 2),
)

### Cómo leer el resultado

Un score alto no demuestra que la predicción sea correcta. ResNet18
debe repartir sus scores entre las categorías de ImageNet incluso
si ninguna describe bien la entrada.

Revisamos tres cosas distintas:

1. **compatibilidad técnica:** el tensor tiene la forma y escala
   esperadas;
2. **plausibilidad:** las alternativas top-k tienen sentido visual;
3. **calidad medible:** las predicciones se comparan con labels
   conocidos en un conjunto independiente.

Solo ejecutamos las dos primeras. ResNet18 también hereda los sesgos
y límites de sus datos. En un dominio propio necesitamos datos
representativos y una evaluación independiente.

---
## Práctica C — detección con YOLO nano

Un **detector** debe reconocer una categoría y localizar cada
instancia. A diferencia del clasificador anterior, puede producir
una lista de longitud variable:

```text
imagen → detector → [
    {clase, score, caja},
    {clase, score, caja},
    ...
]
```

Usaremos **YOLO11n**, una variante nano, solo para inferencia en CPU.
Fue preentrenada con categorías cotidianas; no conoce clases propias
de una empresa si no fue adaptada para ellas.

El modelo devuelve cajas en formato `xyxy`:

- `x1, y1`: esquina superior izquierda;
- `x2, y2`: esquina inferior derecha.

En este notebook las coordenadas son absolutas y se miden en
píxeles. Otros datasets pueden usar centro, ancho y alto, o valores
normalizados entre 0 y 1. Siempre debemos verificar el formato.

YOLO primero produce **candidatos**. Luego intervienen dos decisiones:

```text
candidatos
    ↓ threshold de score: elimina candidatos débiles
cajas restantes
    ↓ NMS: elimina cajas muy superpuestas del mismo objeto
detecciones finales
```

**NMS** (Non-Maximum Suppression) conserva la caja con mejor score
entre propuestas muy solapadas. El threshold controla cuánta
evidencia exigimos antes de mostrar una detección.

El código tiene un fallback explícito: si Ultralytics, sus pesos o
la conexión no están disponibles, no inventa detecciones. La clase
continúa con una lista vacía y la práctica de IoU sigue funcionando.

In [ ]:
imagen_deteccion_nombre = (
    "deteccion_calle"
    if "deteccion_calle" in imagenes
    else "paisaje"
)
imagen_deteccion = imagenes[imagen_deteccion_nombre]

cajas_yolo = np.empty((0, 4), dtype=float)
scores_yolo = np.empty(0, dtype=float)
clases_yolo = np.empty(0, dtype=int)
nombres_yolo = {}
yolo_disponible = False
error_yolo = None

try:
    from ultralytics import YOLO

    cache_modelos = Path("modelos_cache")
    cache_modelos.mkdir(exist_ok=True)
    ruta_yolo = cache_modelos / "yolo11n.pt"
    detector_yolo = YOLO(str(ruta_yolo))
    resultado_yolo = detector_yolo.predict(
        source=imagen_deteccion,
        imgsz=480 if FAST_MODE else 640,
        conf=0.10,
        device="cpu",
        verbose=False,
    )[0]

    cajas_yolo = resultado_yolo.boxes.xyxy.cpu().numpy()
    scores_yolo = resultado_yolo.boxes.conf.cpu().numpy()
    clases_yolo = resultado_yolo.boxes.cls.cpu().numpy().astype(int)
    nombres_yolo = resultado_yolo.names
    yolo_disponible = True
except Exception as error:
    error_yolo = f"{type(error).__name__}: {error}"

print("Imagen:", imagen_deteccion_nombre)
print("YOLO ejecutado:", yolo_disponible)
print("Candidatas con score >= 0.10:", len(cajas_yolo))
if error_yolo:
    print("Fallback sin detecciones:", error_yolo)

In [ ]:
UMBRAL_VISUAL = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(imagen_deteccion)
mostradas = 0
for caja, score, clase in zip(
    cajas_yolo, scores_yolo, clases_yolo
):
    if score < UMBRAL_VISUAL:
        continue
    x1, y1, x2, y2 = caja
    ax.add_patch(Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        color="lime",
        linewidth=2.5,
    ))
    etiqueta = nombres_yolo.get(int(clase), str(clase))
    ax.text(
        x1,
        max(0, y1 - 4),
        f"{etiqueta}: {score:.2f}",
        color="black",
        backgroundcolor="lime",
        fontsize=10,
    )
    mostradas += 1

estado = (
    f"{mostradas} detección(es) con score ≥ {UMBRAL_VISUAL}"
    if yolo_disponible
    else "fallback: YOLO no disponible; no se dibujan cajas"
)
ax.set_title(estado)
ax.axis("off")
plt.tight_layout()
plt.show()

if yolo_disponible and mostradas == 0:
    print(
        "El modelo se ejecutó correctamente, pero ninguna candidata "
        "superó el umbral. Cero detecciones también es una salida."
    )

### Qué cambia al mover el umbral

Para una clase de interés:

- un **verdadero positivo (TP)** es un objeto encontrado
  correctamente;
- un **falso positivo (FP)** es una detección que no corresponde a
  un objeto real;
- un **falso negativo (FN)** es un objeto real que el modelo omitió.

Al mover el threshold suele aparecer un intercambio:

- umbral bajo: aparecen más candidatos, puede bajar la cantidad de
  FN, pero pueden aumentar los FP;
- umbral alto: quedan menos candidatos, pueden bajar los FP, pero
  podemos perder objetos verdaderos y aumentar los FN;
- el umbral adecuado depende del costo de cada tipo de error.

El score no reemplaza una evaluación con **ground truth**, es decir,
con cajas reales anotadas por el equipo responsable de los datos.

---
## 5. Cajas, máscaras y métricas

### Métricas de clasificación

Una **métrica** resume un aspecto del comportamiento, no toda la
calidad del sistema. Para pensar una clase como “positiva” usamos:

| Resultado | Significado |
|---|---|
| Verdadero positivo (TP) | Era positivo y el modelo lo detectó |
| Falso positivo (FP) | Era negativo, pero el modelo dijo positivo |
| Falso negativo (FN) | Era positivo, pero el modelo no lo detectó |
| Verdadero negativo (TN) | Era negativo y el modelo lo descartó |

Con esos conteos construimos:

```text
accuracy  = aciertos totales / ejemplos totales
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
F1        = balance armónico entre precision y recall
```

| Métrica | Pregunta que responde | Atención |
|---|---|---|
| Accuracy | ¿Qué proporción total acertamos? | Puede engañar si una clase domina |
| Precision | Cuando dijimos “positivo”, ¿cuántas veces acertamos? | Penaliza falsos positivos |
| Recall | De los positivos reales, ¿cuántos encontramos? | Penaliza falsos negativos |
| F1 | ¿Qué balance hay entre precision y recall? | No incluye verdaderos negativos |

Ejemplo: si solo 1 de cada 100 imágenes contiene una falla, decir
siempre “sin falla” logra 99 % de accuracy y 0 % de recall para la
falla. Por eso la métrica se elige según el riesgo del problema.

En problemas multiclase hay que indicar cómo se promedia: `macro`
da el mismo peso a cada clase; `weighted` pondera por frecuencia.

In [ ]:
y_real = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])

metricas_clasificacion = pd.Series({
    "accuracy": accuracy_score(y_real, y_pred),
    "precision": precision_score(y_real, y_pred),
    "recall": recall_score(y_real, y_pred),
    "F1": f1_score(y_real, y_pred),
})
metricas_clasificacion.round(3).to_frame("valor")

### IoU: cuánto se superponen dos regiones

**Intersection over Union** compara la intersección con la unión:

```text
IoU = área de intersección / área de unión
```

- `0`: no se superponen;
- `1`: coinciden exactamente.

Para una caja `xyxy`:

```text
ancho = x2 - x1
alto  = y2 - y1
área  = ancho × alto
```

Primero obtenemos el rectángulo compartido por ambas cajas. La
unión es el área de A más el área de B menos esa intersección, que
de otro modo contaríamos dos veces.

```text
unión = área(A) + área(B) - intersección
```

Una clase correcta con IoU bajo puede indicar que el objeto fue
reconocido pero quedó mal localizado.

IoU sirve tanto para cajas como para máscaras. Primero lo calculamos
con dos boxes conocidas.

In [ ]:
def validar_caja(caja):
    x1, y1, x2, y2 = np.asarray(caja, dtype=float)
    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            "La caja debe cumplir x2 > x1 e y2 > y1."
        )
    return np.array([x1, y1, x2, y2], dtype=float)


def iou_cajas(caja_a, caja_b):
    a = validar_caja(caja_a)
    b = validar_caja(caja_b)

    inter_x1 = max(a[0], b[0])
    inter_y1 = max(a[1], b[1])
    inter_x2 = min(a[2], b[2])
    inter_y2 = min(a[3], b[3])

    ancho_inter = max(0.0, inter_x2 - inter_x1)
    alto_inter = max(0.0, inter_y2 - inter_y1)
    interseccion = ancho_inter * alto_inter

    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - interseccion
    return interseccion / union


caja_real = np.array([1.0, 1.0, 7.0, 6.0])
caja_predicha = np.array([2.0, 0.5, 8.0, 5.0])
iou_box = iou_cajas(caja_real, caja_predicha)

fig, ax = plt.subplots(figsize=(6, 5))
for caja, color, etiqueta in [
    (caja_real, "tab:green", "ground truth"),
    (caja_predicha, "tab:orange", "predicción"),
]:
    x1, y1, x2, y2 = caja
    ax.add_patch(Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        color=color,
        linewidth=3,
        label=etiqueta,
    ))
ax.set(xlim=(0, 9), ylim=(7, 0), aspect="equal")
ax.grid(alpha=0.25)
ax.legend()
ax.set_title(f"IoU de cajas = {iou_box:.3f}")
plt.show()

### IoU de máscaras

Una **máscara binaria** tiene la misma altura y ancho que la imagen.
Marca con `True` o 1 los píxeles de una región y con `False` o 0 el
resto.

```text
imagen:  (alto, ancho, 3)
máscara: (alto, ancho)
```

En segmentación semántica puede existir un mapa de enteros donde
cada valor representa una clase. En segmentación de instancias
necesitamos además separar objetos distintos de la misma clase.

El cálculo de IoU es el mismo, pero contamos píxeles en lugar de
áreas de rectángulos. Una métrica relacionada es **Dice**, que da
más peso a la intersección y también vale 1 ante coincidencia total.

In [ ]:
def iou_mascaras(mascara_a, mascara_b):
    a = np.asarray(mascara_a, dtype=bool)
    b = np.asarray(mascara_b, dtype=bool)
    if a.shape != b.shape:
        raise ValueError("Las máscaras deben tener la misma forma.")
    interseccion = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(interseccion / union) if union else 1.0


yy, xx = np.ogrid[:80, :100]
mascara_real = ((xx - 43) / 26) ** 2 + ((yy - 40) / 23) ** 2 <= 1
mascara_predicha = (
    ((xx - 50) / 25) ** 2 + ((yy - 37) / 20) ** 2 <= 1
)
superposicion = (
    mascara_real.astype(int)
    + 2 * mascara_predicha.astype(int)
)
iou_mask = iou_mascaras(mascara_real, mascara_predicha)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, matriz, titulo in zip(
    axes,
    [mascara_real, mascara_predicha, superposicion],
    [
        "Máscara real",
        "Máscara predicha",
        f"Superposición — IoU {iou_mask:.3f}",
    ],
):
    ax.imshow(matriz, cmap="viridis")
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()
plt.show()

### De IoU a AP y mAP

Para evaluar detección no basta con que la clase sea correcta. Una
predicción cuenta como acierto solo si:

- predice la clase adecuada;
- alcanza el mínimo de IoU elegido;
- se asigna a un objeto real todavía no emparejado.

Si se dibujan tres cajas sobre el mismo objeto, una puede ser TP y
las duplicadas se consideran FP. El proceso general es:

1. comparamos cada caja predicha con una caja real mediante IoU;
2. un criterio, por ejemplo `IoU ≥ 0.50`, decide si hay
   correspondencia o **match**;
3. variamos el threshold de score y obtenemos precision/recall;
4. dibujamos una curva precision-recall;
5. **AP** (Average Precision) resume el área de esa curva para una
   clase;
6. **mAP** (mean Average Precision) promedia AP entre clases.

`mAP@0.5` usa IoU 0.50 para decidir matches.
`mAP@0.5:0.95` promedia resultados con umbrales desde 0.50 hasta
0.95, por lo que es más exigente con la localización.

En segmentación también se reportan IoU o Dice por clase. La métrica
correcta depende de la tarea y del costo de los errores. Comparar
números exige además el mismo dataset, split, definición de clases
y protocolo de evaluación.

---
## Actividad — cambiar una decisión y observar tres efectos

Modificá solo las tres variables marcadas. Después respondé:

1. ¿La rotación conserva el significado de la imagen?
2. ¿Qué cambio en la caja sube o baja más el IoU?
3. ¿Cuántas detecciones conserva cada threshold y qué riesgo cambia?

Si YOLO no está disponible o no produjo cajas, el tercer resultado
será cero; explicá por qué eso no invalida la práctica de IoU.

In [ ]:
# TODO: probá ROTACION_ACTIVIDAD=45, otra caja y umbrales 0.10/0.50.
ROTACION_ACTIVIDAD = 10
CAJA_PRED_ACTIVIDAD = np.array([1.5, 1.0, 7.5, 5.5])
UMBRAL_ACTIVIDAD = 0.25

imagen_rotada = TF.rotate(imagen_base, ROTACION_ACTIVIDAD)
iou_actividad = iou_cajas(caja_real, CAJA_PRED_ACTIVIDAD)
detecciones_actividad = int(
    np.sum(scores_yolo >= UMBRAL_ACTIVIDAD)
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(imagen_rotada)
axes[0].set_title(f"Rotación: {ROTACION_ACTIVIDAD}°")
axes[0].axis("off")

axes[1].bar(
    ["IoU caja", "detecciones"],
    [iou_actividad, detecciones_actividad],
    color=["tab:orange", "tab:blue"],
)
axes[1].set_title(
    f"Threshold de detección: {UMBRAL_ACTIVIDAD:.2f}"
)
axes[1].set_ylim(
    0, max(1.0, detecciones_actividad + 0.5)
)
plt.tight_layout()
plt.show()

pd.Series({
    "IoU de la caja": iou_actividad,
    "detecciones sobre el umbral": detecciones_actividad,
    "YOLO disponible": yolo_disponible,
}).to_frame("resultado")

### Límite de esta introducción

Hoy usamos modelos ya entrenados y ejemplos controlados. No podemos
concluir que funcionen para imágenes clínicas, industriales,
satelitales o de dron. Para eso hacen falta:

- definición precisa del objetivo;
- datos y anotaciones representativas;
- separación o **split** entre entrenamiento, validación y test;
- evitar **data leakage**, cuando información de validación o test
  se filtra hacia el entrenamiento;
- entrenamiento o adaptación;
- métricas por clase y análisis de errores;
- validación con condiciones reales de captura.

```text
train      → ajustar parámetros
validation → elegir modelo y decisiones
test       → estimar una vez el resultado final
```

Si fotografías casi idénticas del mismo sitio quedan en train y
test, el resultado puede parecer excelente sin medir verdadera
generalización.

---

## Síntesis de la clase

- Clasificación devuelve clases; detección agrega cajas; segmentación produce máscaras.
- Una CNN aprende features locales a partir de píxeles.
- Preprocesamiento hace compatible la entrada; augmentation agrega variación al train.
- Los modelos preentrenados permiten inferencia y son la base de transfer learning.
- IoU mide superposición; AP y mAP resumen la calidad de un detector.
- ResNet/EfficientNet, YOLO/Faster R-CNN y U-Net/Mask R-CNN/SAM resuelven familias de tareas distintas.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

En la clase 6 cambiaremos de modalidad: representaremos audio y texto, ejecutaremos ASR y veremos attention y KV cache.

## Conexión con los tracks

El Track Imagen profundizará entrenamiento, transfer learning, augmentation de dominio, YOLO/Faster R-CNN, U-Net/Mask R-CNN/SAM, mAP en datasets reales y análisis de errores.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.